# Paper-style results table + figures (DA-MAPPO Table V format), inference only

**2026-10-05.** Teacher's feedback (`sessions/2026-10-05.md`): present the
comparison the way the base paper does. DA-MAPPO (`bin/91`) reports results
as tables (Tables IV/V/VI: R_success, R_collision, R_timeout, T_ave, L_ave)
plus clean white-background bar charts (their Fig. 6). This notebook
produces **both** from our already-trained models:

- **Tables**, same columns as DA-MAPPO's Table V
- **Figures**, white background, grouped bars with value labels

**No training happens here.** Every `final_model.pt` is loaded and
evaluated (inference only, CPU is fine, no Kaggle needed).

## Protocol (follows `docs/plans/02_experiment_protocol.md`)

- **200 evaluation episodes** per (method, training seed)
- **Deterministic policy**: the actor's mean action, no sampling
- **Disjoint evaluation seed** (`EVAL_SEED=7001`): never a training seed
  (42-46), never the training-time eval seed (999), never the
  generalization-test seeds (2001-2005). The environment's random stream
  does not depend on the agent's actions, so **every method sees the
  identical 200 episode layouts** (paired comparison).
- Stage 2: mean +/- std over the 5 training seeds. Stage 3: single seed
  (42), so the uncertainty shown is a 95% Wilson interval over the 200
  episodes, not a seed-to-seed std.

## Metric definitions (matching DA-MAPPO's)

| Column | Meaning |
|---|---|
| R_success | % of episodes where every drone reached its target with no collision |
| R_collision | % of episodes with any drone-drone or drone-obstacle collision |
| R_timeout | % of episodes that ended by time limit (neither of the above) |
| T_ave | mean decision steps to finish, **successful episodes only** |
| L_ave | mean path length **per drone**, successful episodes only, in world units |

`L_ave`: DA-MAPPO's write-up says "total distance by all UAVs"; we report
the per-drone mean (the total is stored too, in the raw JSON). World units
and drone counts differ from the paper, so `T_ave`/`L_ave` are for
comparing our own methods with each other, not with the paper's numbers.

## Important: these numbers differ from the training-log numbers

Training-time evaluation during all earlier runs sampled **stochastic**
actions on 20-episode batches. This notebook uses the deterministic policy
on 200 episodes. Both are consistent *within* themselves, but they are not
interchangeable (`sessions/2026-09-30.md`, generalization test).

## Models evaluated

- **Stage 2** (5 drones, 5 obstacles, moving targets), seeds 42-46:
  B4 (fixed alpha=0.5), H (hand-coded tau-rule), M (prior + learned residual)
- **Stage 3** (8 drones, 10 obstacles), seed 42 only: B4, M, and
  **M without Hungarian** (assignment fixed at reset, evaluated in the same
  fixed-assignment environment it was trained in)

Results are cached per model in `output/raw/`, so re-running resumes where
it stopped.


In [ ]:
# Cell 1 -- Imports
import os, json, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal
from scipy.optimize import linear_sum_assignment

torch.set_num_threads(1)   # tiny networks: one thread is faster than many
DEVICE = torch.device('cpu')
print('PyTorch', torch.__version__, '| device', DEVICE)


In [ ]:
# Cell 2 — Conflict Graph
# Ported from code/algorithms/conflict_graph.py

import numpy as np

def _time_to_cpa(pos_i, pos_j, vel_i, vel_j):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return np.inf
    return -np.dot(p, v) / vv

def _dist_at_cpa(pos_i, pos_j, vel_i, vel_j, horizon):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return float(np.linalg.norm(p)), 0.0
    t_star = -np.dot(p, v) / vv
    t_clamped = float(np.clip(t_star, 0.0, horizon))
    return float(np.linalg.norm(p + v * t_clamped)), t_star

class ConflictGraph:
    K_NBR = 4

    def __init__(self, n_drones, horizon=3.0, d_danger=9.0):
        self.n      = n_drones
        self.H      = horizon
        self.d_dan  = d_danger
        self.adj    = np.zeros((n_drones, n_drones), dtype=bool)
        self.t_mat  = np.full((n_drones, n_drones), np.inf)

    def update(self, pos, vel):
        self.adj[:] = False
        self.t_mat[:] = np.inf
        for i in range(self.n):
            for j in range(i + 1, self.n):
                dcpa, t_star = _dist_at_cpa(pos[i], pos[j], vel[i], vel[j], self.H)
                self.t_mat[i, j] = self.t_mat[j, i] = t_star
                if dcpa < self.d_dan and 0.0 <= t_star <= self.H:
                    self.adj[i, j] = self.adj[j, i] = True

    def n_conflict(self, i):
        return int(self.adj[i].sum())

    def tau_collision(self, i):
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return self.H
        return float(np.min(self.t_mat[i, nbrs]))

    def neighbor_obs(self, i, pos, vel):
        obs = np.zeros(self.K_NBR * 5, dtype=np.float32)
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return obs
        order = np.argsort(self.t_mat[i, nbrs])
        nbrs  = nbrs[order]
        for slot, j in enumerate(nbrs[:self.K_NBR]):
            b = slot * 5
            obs[b:b+2]   = pos[j] - pos[i]
            obs[b+2:b+4] = vel[j] - vel[i]
            obs[b+4]     = 1.0
        return obs

In [ ]:
# Cell 3 -- Environment (Stage 3 base + Hungarian-assignment ablation switch)
#
# Same ablation switch as code/notebooks/v3/stage2/ablation-hungarian/ --
# ported here identically since Stage 2 and Stage 3's environment code is
# byte-identical (only CFG values like n_drones/n_obstacles differ). See
# Cell 0 for why this needs testing at Stage 3 specifically, not just
# Stage 2.

import numpy as np
from scipy.optimize import linear_sum_assignment

K_NBR    = 4
BASE_DIM = 10
OBS_DIM  = BASE_DIM + K_NBR * 5  # 30

class MultiUAVEnv:
    def __init__(self, n_drones=5, n_obstacles=5, world_size=500.0,
                 max_speed=5.0, max_steps=500, collision_radius=3.0,
                 target_radius=25.0, success_bonus=20.0,
                 target_speed=1.0, use_hungarian=True, seed=None):
        self.n         = n_drones
        self.n_obs     = n_obstacles
        self.W         = world_size
        self.max_sp    = max_speed
        self.max_steps = max_steps
        self.col_r     = collision_radius
        self.tgt_r     = target_radius
        self.bonus     = success_bonus
        self.tgt_speed = target_speed
        # Ablation switch (see Cell 0): True = per-step optimal Hungarian
        # assignment (normal). False = fixed pairing decided once at
        # reset(), never re-solved.
        self.use_hungarian = use_hungarian
        self.rng       = np.random.default_rng(seed)
        self.cg        = ConflictGraph(n_drones, horizon=3.0,
                                       d_danger=collision_radius * 3.0)
        self.obs_dim   = OBS_DIM
        self.act_dim   = 2

    def reset(self):
        self.t                  = 0
        self.collision_occurred = False
        all_pos = self._sample_non_overlapping(self.n * 2, self.col_r * 2)
        self.drone_pos  = all_pos[:self.n].copy()
        self.target_pos = all_pos[self.n:].copy()
        self.obstacle_pos = self._place_obstacles(
            np.concatenate([self.drone_pos, self.target_pos]), self.n_obs
        )
        self.drone_vel = np.zeros((self.n, 2), dtype=np.float32)
        angles = self.rng.uniform(0, 2 * np.pi, self.n)
        self.target_vel = (self.tgt_speed * np.stack(
            [np.cos(angles), np.sin(angles)], axis=1
        )).astype(np.float32)
        # Ablation: fixed pairing (arbitrary) when Hungarian is off -- never
        # re-solved, even though targets keep moving. See Cell 0.
        self.assignment = self._hungarian() if self.use_hungarian else np.arange(self.n)
        self.cg.update(self.drone_pos, self.drone_vel)
        return self._obs()

    def step(self, actions):
        self.t += 1
        actions = np.clip(actions, -self.max_sp, self.max_sp)
        self.drone_vel = actions.astype(np.float32)
        self.drone_pos = np.clip(self.drone_pos + self.drone_vel, 0.0, self.W)

        self.target_pos = self.target_pos + self.target_vel
        for k in range(self.n):
            for dim in range(2):
                if self.target_pos[k, dim] < 10.0:
                    self.target_pos[k, dim] = 10.0
                    self.target_vel[k, dim] *= -1.0
                elif self.target_pos[k, dim] > self.W - 10.0:
                    self.target_pos[k, dim] = self.W - 10.0
                    self.target_vel[k, dim] *= -1.0

        # Ablation: keep the reset()-time assignment fixed when Hungarian is
        # off -- re-solved every step only in the normal (True) case.
        if self.use_hungarian:
            self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)

        rewards, r_mission, r_safety = self._rewards()
        all_reached   = self._all_reached()
        any_collision = self._any_collision()
        timeout       = self.t >= self.max_steps

        if any_collision:
            self.collision_occurred = True

        terminated = all_reached or any_collision
        truncated  = timeout

        if all_reached and self.bonus:
            rewards   = rewards + self.bonus
            r_mission = r_mission + self.bonus   # bonus is a mission outcome, not safety

        # PAH inputs — per drone, from the conflict graph + current assignment.
        pah_inputs = {
            'tau': np.array([self.cg.tau_collision(i) for i in range(self.n)], dtype=np.float32),
            'd_target': np.array([
                np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]])
                for i in range(self.n)
            ], dtype=np.float32),
            'n_conflict': np.array([self.cg.n_conflict(i) for i in range(self.n)], dtype=np.float32),
        }

        info = {
            'all_targets_reached': all_reached,
            'any_collision':       any_collision,
            'episode_collision':   self.collision_occurred,
            'r_mission':           r_mission,
            'r_safety':            r_safety,
            'pah_inputs':          pah_inputs,
        }
        return self._obs(), rewards, terminated, truncated, info

    def _obs(self):
        obs = np.zeros((self.n, self.obs_dim), dtype=np.float32)
        for i in range(self.n):
            rel = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            cl  = self._clearances(i)
            base = np.concatenate([
                self.drone_pos[i] / self.W,
                self.drone_vel[i] / self.max_sp,
                rel               / self.W,
                cl,
            ])
            nbr = self.cg.neighbor_obs(i, self.drone_pos, self.drone_vel)
            for s in range(K_NBR):
                nbr[s*5 : s*5+2]   /= self.W
                nbr[s*5+2 : s*5+4] /= self.max_sp
            obs[i] = np.concatenate([base, nbr])
        return obs

    def _rewards(self):
        r_mission = np.zeros(self.n, dtype=np.float32)
        r_safety  = np.zeros(self.n, dtype=np.float32)
        d_danger  = self.col_r * 3.0
        zone_w    = d_danger - self.col_r
        for i in range(self.n):
            rel  = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            dist = np.linalg.norm(rel)
            direction  = rel / (dist + 1e-6)
            r_progress = float(np.dot(self.drone_vel[i], direction) / self.max_sp)
            r_mission[i] = 0.4 * r_progress + 0.3 * (-dist / self.W)
            min_d = float('inf')
            for j in range(self.n):
                if j != i:
                    min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]))
            for op in self.obstacle_pos:
                min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - op))
            if min_d < self.col_r:
                r_safety[i] = -1.0
            elif min_d < d_danger:
                r_safety[i] = -((d_danger - min_d) / zone_w)
        rewards = r_mission + 0.3 * r_safety
        return rewards, r_mission, r_safety

    def _hungarian(self):
        cost = np.array([
            [np.linalg.norm(self.drone_pos[i] - self.target_pos[j])
             for j in range(self.n)]
            for i in range(self.n)
        ])
        _, col = linear_sum_assignment(cost)
        return col

    def _all_reached(self):
        return all(
            np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]]) <= self.tgt_r
            for i in range(self.n)
        )

    def _any_collision(self):
        for i in range(self.n):
            for j in range(self.n):
                if j != i and np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]) < self.col_r:
                    return True
            for op in self.obstacle_pos:
                if np.linalg.norm(self.drone_pos[i] - op) < self.col_r:
                    return True
        return False

    def _clearances(self, i):
        pos = self.drone_pos[i]
        cl  = np.array([self.W - pos[1], pos[1], self.W - pos[0], pos[0]], dtype=np.float32)
        for op in self.obstacle_pos:
            diff = op - pos
            if diff[1] > 0: cl[0] = min(cl[0],  diff[1])
            else:           cl[1] = min(cl[1], -diff[1])
            if diff[0] > 0: cl[2] = min(cl[2],  diff[0])
            else:           cl[3] = min(cl[3], -diff[0])
        return cl / self.W

    def _sample_non_overlapping(self, n, min_dist):
        pts, attempts = [], 0
        while len(pts) < n:
            attempts += 1
            if attempts > 10000:
                raise RuntimeError('World too crowded.')
            c = self.rng.uniform(10.0, self.W - 10.0, 2)
            if not any(np.linalg.norm(c - p) < min_dist for p in pts):
                pts.append(c)
        return np.array(pts, dtype=np.float32)

    def _place_obstacles(self, existing, n_obs, max_tries=1000):
        if n_obs <= 0:
            return np.zeros((0, 2), dtype=np.float32)
        r_min = 2.0 * self.col_r
        placed = []
        for _ in range(n_obs):
            for _ in range(max_tries):
                c = self.rng.uniform(10.0, self.W - 10.0, 2)
                if any(np.linalg.norm(c - o) < r_min for o in placed): continue
                if any(np.linalg.norm(c - p) < self.col_r for p in existing): continue
                placed.append(c)
                break
            else:
                break
        return np.array(placed, dtype=np.float32) if placed else np.zeros((0, 2), dtype=np.float32)

print('obs_dim = {}  (10 base + {}x5 conflict neighbors, normalized)'.format(OBS_DIM, K_NBR))
print('step() now also returns info["pah_inputs"] = {tau, d_target, n_conflict}')


In [ ]:
# Cell 4 -- Actor (inference only)
#
# Only the ACTOR is needed to evaluate a trained policy: the critic and PAH
# shape training, they play no part in choosing actions at test time.
# Architecture is identical to every Stage 2 / Stage 3 notebook.

class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu_head = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

    def forward(self, x):
        mu      = self.mu_head(self.net(x))
        log_std = self.log_std.clamp(-4.0, 1.5)
        return Normal(mu, log_std.exp().expand_as(mu))


def load_actor(path):
    ckpt  = torch.load(path, map_location=DEVICE, weights_only=True)
    actor = Actor(OBS_DIM, 2).to(DEVICE)
    actor.load_state_dict(ckpt['actor'])
    actor.eval()
    return actor


@torch.no_grad()
def deterministic_action(actor, obs_np):
    """Distribution mean, no sampling (protocol: deterministic evaluation)."""
    return actor(torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)).mean.numpy()

print('Actor ready (deterministic, inference only).')


In [ ]:
# Cell 5 -- Config + checkpoint registry

N_EVAL    = 200      # episodes per (method, seed) -- protocol value
EVAL_SEED = 7001     # disjoint from training (42-46), training-eval (999), generalization (2001-2005)

V3 = '..'            # this notebook lives in code/notebooks/v3/results-table/

S2 = dict(n_drones=5, n_obstacles=5)
S3 = dict(n_drones=8, n_obstacles=10)

def _s2_b4(seed):
    # seed 42 is the base notebook's output; seeds 43-46 have their own folders
    sub = 'output' if seed == 42 else 'seed-%d/output' % seed
    return V3 + '/stage2/stage2-pah-v0/fixed-alpha/' + sub + '/final_model.pt'

REGISTRY = []
for seed in (42, 43, 44, 45, 46):
    REGISTRY += [
        dict(stage=2, method='B4', seed=seed, use_hungarian=True, path=_s2_b4(seed), **S2),
        dict(stage=2, method='H',  seed=seed, use_hungarian=True,
             path=V3 + '/stage2/stage2-pah-v3-pah-change/h-baseline/output-seed%d/final_model.pt' % seed, **S2),
        dict(stage=2, method='M',  seed=seed, use_hungarian=True,
             path=V3 + '/stage2/stage2-pah-v3-pah-change/var-alpha/output-seed%d/final_model.pt' % seed, **S2),
    ]
REGISTRY += [
    dict(stage=3, method='B4',            seed=42, use_hungarian=True,  path=V3 + '/stage3/fixed-alpha/output/final_model.pt',        **S3),
    dict(stage=3, method='M',             seed=42, use_hungarian=True,  path=V3 + '/stage3/var-alpha/output/final_model.pt',          **S3),
    dict(stage=3, method='M-noHungarian', seed=42, use_hungarian=False, path=V3 + '/stage3/ablation-hungarian/output/final_model.pt', **S3),
]

missing = [r['path'] for r in REGISTRY if not os.path.exists(r['path'])]
if missing:
    raise FileNotFoundError('Missing checkpoints:\n  ' + '\n  '.join(missing))

OUT = 'output'
os.makedirs(OUT + '/raw', exist_ok=True)
print('{} models registered, all checkpoints found. N_EVAL={}, EVAL_SEED={}'.format(len(REGISTRY), N_EVAL, EVAL_SEED))


In [ ]:
# Cell 6 -- Evaluate every model (cached, resumable)

def evaluate(entry, n_episodes=N_EVAL, eval_seed=EVAL_SEED):
    """Runs n_episodes deterministic episodes and returns one record each.
    Outcome rule (same as the training-time eval): collision if any
    collision happened during the episode; else success if all targets were
    reached; else timeout."""
    actor = load_actor(entry['path'])
    env = MultiUAVEnv(n_drones=entry['n_drones'], n_obstacles=entry['n_obstacles'],
                      success_bonus=0.0, use_hungarian=entry['use_hungarian'], seed=eval_seed)
    records = []
    for _ in range(n_episodes):
        obs  = env.reset()
        path = np.zeros(entry['n_drones'])
        done = False
        while not done:
            prev = env.drone_pos.copy()
            obs, _, terminated, truncated, info = env.step(deterministic_action(actor, obs))
            path += np.linalg.norm(env.drone_pos - prev, axis=1)
            done = terminated or truncated
        if info['episode_collision']:
            outcome = 'collision'
        elif info['all_targets_reached']:
            outcome = 'success'
        else:
            outcome = 'timeout'
        records.append(dict(outcome=outcome, steps=int(env.t),
                            path_per_drone=float(path.mean()), path_total=float(path.sum())))
    return records


def raw_file(e):
    return '{}/raw/s{}_{}_seed{}.json'.format(OUT, e['stage'], e['method'], e['seed'])


def run_all(only=None):
    """only: optional predicate on a registry entry, to run a subset."""
    for e in REGISTRY:
        if only is not None and not only(e):
            continue
        f = raw_file(e)
        if os.path.exists(f):
            print('cached   ', f)
            continue
        t0 = time.time()
        recs = evaluate(e)
        json.dump(dict(entry=e, eval_seed=EVAL_SEED, n_episodes=len(recs), records=recs), open(f, 'w'))
        succ = 100 * sum(r['outcome'] == 'success' for r in recs) / len(recs)
        print('done {:>5.0f}s  stage{} {:<15} seed{}  success {:5.1f}%'.format(
            time.time() - t0, e['stage'], e['method'], e['seed'], succ))

run_all()


In [ ]:
# Cell 7 -- Aggregate into DA-MAPPO-style tables

def load_results():
    out = {}
    for e in REGISTRY:
        d = json.load(open(raw_file(e)))
        out[(e['stage'], e['method'], e['seed'])] = d['records']
    return out

RESULTS = load_results()

def rates(records):
    n = len(records)
    succ = [r for r in records if r['outcome'] == 'success']
    return dict(
        n=n,
        success   = 100 * len(succ) / n,
        collision = 100 * sum(r['outcome'] == 'collision' for r in records) / n,
        timeout   = 100 * sum(r['outcome'] == 'timeout'   for r in records) / n,
        t_ave = float(np.mean([r['steps'] for r in succ])) if succ else float('nan'),
        l_ave = float(np.mean([r['path_per_drone'] for r in succ])) if succ else float('nan'),
    )

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

def ms(vals):
    return float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

# ---- Stage 2: mean +/- std over 5 training seeds ---------------------------
METHODS_S2 = ['B4', 'H', 'M']
SEEDS_S2   = (42, 43, 44, 45, 46)
S2_AGG, S2_PER_SEED = {}, []
for m in METHODS_S2:
    per = [rates(RESULTS[(2, m, s)]) for s in SEEDS_S2]
    for s, r in zip(SEEDS_S2, per):
        S2_PER_SEED.append((m, s, r))
    S2_AGG[m] = {k: ms([r[k] for r in per]) for k in ('success', 'collision', 'timeout', 't_ave', 'l_ave')}

# ---- Stage 3: single seed, Wilson 95% CI over episodes -----------------------
METHODS_S3 = ['B4', 'M', 'M-noHungarian']
S3_AGG = {}
for m in METHODS_S3:
    recs = RESULTS[(3, m, 42)]
    r = rates(recs)
    k = sum(x['outcome'] == 'success' for x in recs)
    kc = sum(x['outcome'] == 'collision' for x in recs)
    r['success_ci']   = wilson(k, len(recs))
    r['collision_ci'] = wilson(kc, len(recs))
    S3_AGG[m] = r

LABEL = {'B4': 'B4 (fixed alpha)', 'H': 'H (tau-formula)', 'M': 'M (PAH residual)',
         'M-noHungarian': 'M, no Hungarian'}

def fmt(v):  return '{:.1f} +/- {:.1f}'.format(*v)

md2 = ['| Method | R_success (%) | R_collision (%) | R_timeout (%) | T_ave (steps) | L_ave (units/drone) |',
       '|---|---|---|---|---|---|']
for m in METHODS_S2:
    a = S2_AGG[m]
    md2.append('| {} | {} | {} | {} | {} | {} |'.format(
        LABEL[m], fmt(a['success']), fmt(a['collision']), fmt(a['timeout']), fmt(a['t_ave']), fmt(a['l_ave'])))
TABLE_S2 = '\n'.join(md2)

md3 = ['| Method | R_success (%) [95% CI] | R_collision (%) [95% CI] | R_timeout (%) | T_ave (steps) | L_ave (units/drone) |',
       '|---|---|---|---|---|---|']
for m in METHODS_S3:
    r = S3_AGG[m]
    md3.append('| {} | {:.1f} [{:.1f}, {:.1f}] | {:.1f} [{:.1f}, {:.1f}] | {:.1f} | {:.1f} | {:.1f} |'.format(
        LABEL[m], r['success'], *r['success_ci'], r['collision'], *r['collision_ci'], r['timeout'], r['t_ave'], r['l_ave']))
TABLE_S3 = '\n'.join(md3)

print('STAGE 2  (5 drones, 5 obstacles) -- mean +/- std over 5 training seeds, {} episodes each'.format(N_EVAL))
print(TABLE_S2); print()
print('STAGE 3  (8 drones, 10 obstacles) -- seed 42 only, {} episodes'.format(N_EVAL))
print(TABLE_S3)

# Welch t-test across the 5 seed-level success rates (protocol decision rule)
from scipy import stats
print()
for a, b in (('M', 'B4'), ('M', 'H'), ('H', 'B4')):
    xa = [r['success'] for (m, s, r) in S2_PER_SEED if m == a]
    xb = [r['success'] for (m, s, r) in S2_PER_SEED if m == b]
    t, p = stats.ttest_ind(xa, xb, equal_var=False)
    print('Stage 2 success, {} vs {}: diff {:+.2f} pts, Welch p = {:.3f}'.format(a, b, np.mean(xa) - np.mean(xb), p))

with open(OUT + '/table_stage2.md', 'w') as f: f.write(TABLE_S2 + '\n')
with open(OUT + '/table_stage3.md', 'w') as f: f.write(TABLE_S3 + '\n')
with open(OUT + '/per_seed_stage2.csv', 'w') as f:
    f.write('method,seed,success,collision,timeout,t_ave,l_ave\n')
    for m, s, r in S2_PER_SEED:
        f.write('{},{},{:.2f},{:.2f},{:.2f},{:.2f},{:.2f}\n'.format(m, s, r['success'], r['collision'], r['timeout'], r['t_ave'], r['l_ave']))
print('\nSaved tables to', OUT)


In [ ]:
# Cell 8 -- Figures (white background, DA-MAPPO Fig. 6 style: grouped bars, value labels)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.alpha': 0.25, 'grid.linestyle': '-',
})
GREEN, RED, GREY = '#2E7D32', '#C62828', '#546E7A'


def grouped_bars(ax, groups, series, ylabel, title, ylim, fmt_str='{:.1f}', err=None):
    """groups: x labels; series: list of (name, values, colour); err: list of (lo, hi) per series or None."""
    n = len(series); w = 0.8 / n
    x = np.arange(len(groups))
    for i, (name, vals, colour) in enumerate(series):
        pos = x + (i - (n - 1) / 2) * w
        yerr = None
        if err is not None and err[i] is not None:
            yerr = np.array(err[i])
        bars = ax.bar(pos, vals, w * 0.92, label=name, color=colour, edgecolor='white', yerr=yerr,
                      error_kw=dict(ecolor='black', capsize=3, lw=1))
        if yerr is None:
            upper = np.zeros(len(vals))
        elif yerr.ndim == 2:
            upper = yerr[1]
        else:
            upper = yerr
        for b, v, u in zip(bars, vals, upper):
            ax.text(b.get_x() + b.get_width() / 2, v + u + (ylim[1] - ylim[0]) * 0.012,
                    fmt_str.format(v), ha='center', va='bottom', fontsize=8.5)
    ax.set_xticks(x); ax.set_xticklabels(groups)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_ylim(*ylim)


# ---- Fig 1: Stage 2 success / collision -----------------------------------
fig, ax = plt.subplots(figsize=(6.6, 4.2))
grouped_bars(ax, [LABEL[m] for m in METHODS_S2],
    [('R_success',   [S2_AGG[m]['success'][0]   for m in METHODS_S2], GREEN),
     ('R_collision', [S2_AGG[m]['collision'][0] for m in METHODS_S2], RED)],
    'Rate (%)', 'Stage 2 (5 drones, 5 obstacles): mean +/- std over 5 seeds', (0, 112),
    err=[[S2_AGG[m]['success'][1]   for m in METHODS_S2],
         [S2_AGG[m]['collision'][1] for m in METHODS_S2]])
ax.legend(loc='upper right', frameon=False, ncol=2)
plt.tight_layout(); fig.savefig(OUT + '/fig1_stage2_success_collision.png', dpi=200); fig.savefig(OUT + '/fig1_stage2_success_collision.svg'); plt.close(fig)

# ---- Fig 2: Stage 3 success / collision (95% CI) -------------------------
fig, ax = plt.subplots(figsize=(6.6, 4.2))
s_vals = [S3_AGG[m]['success'] for m in METHODS_S3]; c_vals = [S3_AGG[m]['collision'] for m in METHODS_S3]
s_err  = [[S3_AGG[m]['success'] - S3_AGG[m]['success_ci'][0] for m in METHODS_S3],
          [S3_AGG[m]['success_ci'][1] - S3_AGG[m]['success'] for m in METHODS_S3]]
c_err  = [[S3_AGG[m]['collision'] - S3_AGG[m]['collision_ci'][0] for m in METHODS_S3],
          [S3_AGG[m]['collision_ci'][1] - S3_AGG[m]['collision'] for m in METHODS_S3]]
grouped_bars(ax, [LABEL[m] for m in METHODS_S3],
    [('R_success', s_vals, GREEN), ('R_collision', c_vals, RED)],
    'Rate (%)', 'Stage 3 (8 drones, 10 obstacles): seed 42, 95% CI over 200 episodes', (0, 112), err=[s_err, c_err])
ax.legend(loc='upper right', frameon=False, ncol=2)
plt.tight_layout(); fig.savefig(OUT + '/fig2_stage3_success_collision.png', dpi=200); fig.savefig(OUT + '/fig2_stage3_success_collision.svg'); plt.close(fig)

# ---- Fig 3 / 4: T_ave and L_ave --------------------------------------------
def tl_figure(methods, get_t, get_l, title, fname, t_err=None, l_err=None):
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0))
    g = [LABEL[m] for m in methods]
    grouped_bars(axes[0], g, [('T_ave', [get_t(m) for m in methods], GREEN)], 'Decision steps', 'T_ave (successful episodes)',
                 (0, max(get_t(m) for m in methods) * 1.25), err=None if t_err is None else [t_err])
    grouped_bars(axes[1], g, [('L_ave', [get_l(m) for m in methods], GREEN)], 'Path length per drone (units)', 'L_ave (successful episodes)',
                 (0, max(get_l(m) for m in methods) * 1.25), err=None if l_err is None else [l_err])
    for a in axes: a.tick_params(axis='x', labelsize=8.5)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout(); fig.savefig(OUT + '/' + fname + '.png', dpi=200); fig.savefig(OUT + '/' + fname + '.svg'); plt.close(fig)

tl_figure(METHODS_S2, lambda m: S2_AGG[m]['t_ave'][0], lambda m: S2_AGG[m]['l_ave'][0],
          'Stage 2: time and path length (mean +/- std over 5 seeds)', 'fig3_stage2_time_path',
          t_err=[S2_AGG[m]['t_ave'][1] for m in METHODS_S2], l_err=[S2_AGG[m]['l_ave'][1] for m in METHODS_S2])
tl_figure(METHODS_S3, lambda m: S3_AGG[m]['t_ave'], lambda m: S3_AGG[m]['l_ave'],
          'Stage 3: time and path length (seed 42)', 'fig4_stage3_time_path')

# ---- Fig 5: effect sizes, both measured in THIS notebook -----------------------
alpha_spread    = max(S2_AGG[m]['success'][0] for m in METHODS_S2) - min(S2_AGG[m]['success'][0] for m in METHODS_S2)
alpha_spread_s3 = abs(S3_AGG['B4']['success'] - S3_AGG['M']['success'])
hung_drop       = S3_AGG['M']['success'] - S3_AGG['M-noHungarian']['success']
fig, ax = plt.subplots(figsize=(8.0, 4.2))
vals = [alpha_spread, alpha_spread_s3, hung_drop]
bars = ax.bar(['How alpha is chosen\n(B4 vs H vs M)\nStage 2, 5 seeds',
               'How alpha is chosen\n(B4 vs M)\nStage 3, seed 42',
               'Removing optimal assignment\n(M vs M no Hungarian)\nStage 3, seed 42'],
              vals, 0.55, color=[GREY, GREY, GREEN], edgecolor='white')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.6, '{:.1f} pts'.format(v), ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Change in R_success (percentage points)')
ax.set_title('Size of each effect on success rate'); ax.set_ylim(0, max(vals) * 1.25)
plt.tight_layout(); fig.savefig(OUT + '/fig5_effect_sizes.png', dpi=200); fig.savefig(OUT + '/fig5_effect_sizes.svg'); plt.close(fig)

print('Saved fig1-fig5 (.png + .svg) to', OUT)
print('alpha-source spread: Stage 2 {:.2f} pts, Stage 3 {:.2f} pts | Hungarian removal (Stage 3, seed 42): {:.2f} pts'.format(
    alpha_spread, alpha_spread_s3, hung_drop))


In [ ]:
# Cell 9 -- Training curves in the style of IGAT-MARL's Fig. 3 (one panel per condition)
#
# Reads each run's history.json (written during training). IMPORTANT: these are
# TRAINING-TIME evaluations (stochastic policy, 20 episodes per checkpoint), so
# the levels differ from the deterministic 200-episode table above. They show
# HOW each method learned, not the final scores. Stage 2 curves are the mean of
# 5 seeds with a +/-1 std band; Stage 3 has one seed per method, so its band is
# a 95% binomial interval of the smoothed estimate (not a seed-to-seed spread).

WINDOW = 5                      # moving-average window, in checkpoints (about 500 episodes)
GRID   = np.arange(100, 7901, 100)

def hist_path(entry):
    return os.path.join(os.path.dirname(entry['path']), 'history.json')

def series(entry, key):
    h = json.load(open(hist_path(entry)))
    y = np.interp(GRID, h['episode'], [100.0 * v if key != 'entropy' else v for v in h[key]])
    return y

def trailing_mean(y, w=WINDOW):
    return np.array([y[max(0, i - w + 1): i + 1].mean() for i in range(len(y))])

def entries(stage, method):
    return [e for e in REGISTRY if e['stage'] == stage and e['method'] == method]

def curve(stage, method, key):
    """Returns (mean_line, lo, hi) on GRID."""
    es = entries(stage, method)
    sm = np.array([trailing_mean(series(e, key)) for e in es])
    if len(es) > 1:                                  # Stage 2: across seeds
        m, sd = sm.mean(axis=0), sm.std(axis=0, ddof=1)
        return m, m - sd, m + sd
    m = sm[0]                                        # Stage 3: single seed
    if key == 'entropy':
        return m, None, None
    n_eff = 20 * np.minimum(np.arange(1, len(GRID) + 1), WINDOW)
    p = np.clip(m / 100.0, 1e-6, 1 - 1e-6)
    half = 100 * 1.96 * np.sqrt(p * (1 - p) / n_eff)
    return m, np.clip(m - half, 0, 100), np.clip(m + half, 0, 100)

STYLE = {'B4': ('Benchmark (B4, fixed alpha)', '#808080'),
         'H':  ('H (tau-formula)',            '#D98E2B'),
         'M':  ('M (PAH residual, ours)',     '#3B6FB6'),
         'M-noHungarian': ('M, no Hungarian (ablation)', '#C0392B')}

PANELS = [
    (2, ['B4', 'H', 'M'],              'success',   'Success rate (%)',   'Success Rate for N=5 UAVs (Stage 2)',   (0, 100), '(a) Stage 2, 5 drones: success rate'),
    (2, ['B4', 'H', 'M'],              'collision', 'Collision rate (%)', 'Collision Rate for N=5 UAVs (Stage 2)', (0, 40),  '(b) Stage 2, 5 drones: collision rate'),
    (3, ['B4', 'M', 'M-noHungarian'],  'success',   'Success rate (%)',   'Success Rate for N=8 UAVs (Stage 3)',   (0, 100), '(c) Stage 3, 8 drones: success rate'),
    (3, ['B4', 'M', 'M-noHungarian'],  'collision', 'Collision rate (%)', 'Collision Rate for N=8 UAVs (Stage 3)', (0, 70),  '(d) Stage 3, 8 drones: collision rate'),
    (2, ['B4', 'H', 'M'],              'entropy',   'Policy entropy',     'Policy Entropy for N=5 UAVs (Stage 2)', None,     '(e) Stage 2, 5 drones: policy entropy'),
    (3, ['B4', 'M', 'M-noHungarian'],  'entropy',   'Policy entropy',     'Policy Entropy for N=8 UAVs (Stage 3)', None,     '(f) Stage 3, 8 drones: policy entropy'),
]

with plt.rc_context({'axes.spines.top': True, 'axes.spines.right': True, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 8.5, 'axes.titlesize': 9, 'axes.labelsize': 8}):
    fig, axes = plt.subplots(3, 2, figsize=(9.6, 11.0))
    for ax, (stage, methods, key, ylab, title, ylim, caption) in zip(axes.ravel(), PANELS):
        for m in methods:
            label, colour = STYLE[m]
            mean, lo, hi = curve(stage, m, key)
            if lo is not None:
                ax.fill_between(GRID, lo, hi, color=colour, alpha=0.18, linewidth=0)
            ax.plot(GRID, mean, color=colour, lw=1.6, label=label)
        if stage == 3:
            ax.axvline(500, color='black', lw=0.8, ls=':')
            if key != 'entropy':      # label only where it does not sit under a legend or curve
                ax.text(560, ax.get_ylim()[0] + 0.03 * (ax.get_ylim()[1] - ax.get_ylim()[0]), 'unfreeze', fontsize=7, rotation=90, va='bottom')
        if ylim: ax.set_ylim(*ylim)
        ax.set_xlim(0, 8000)
        ax.set_xlabel('Episodes'); ax.set_ylabel(ylab); ax.set_title(title)
        ax.legend(fontsize=7, loc='best', framealpha=0.9)
        ax.text(0.5, -0.30, caption, transform=ax.transAxes, ha='center', va='top', fontsize=10, family='serif')
    fig.suptitle('Training curves (training-time evaluation: stochastic policy, 20 episodes per checkpoint, '
                 '5-checkpoint moving average)\nStage 2: mean +/- 1 std over 5 seeds.  Stage 3: one seed, band = 95% binomial interval.  '
                 'Dotted line = actor and PAH unfrozen (Stage 3).', fontsize=9, y=0.997)
    fig.tight_layout(rect=(0, 0, 1, 0.972), h_pad=3.2)
    fig.savefig(OUT + '/fig6_training_curves.png', dpi=200)
    fig.savefig(OUT + '/fig6_training_curves.svg')
    plt.close(fig)
print('Saved fig6_training_curves (.png + .svg) to', OUT)


In [ ]:
# Cell 10 -- Comparison with the base paper (DA-MAPPO, bin/91 Table V, dynamic)
#
# Teacher's ask: compare OUR results directly against the base paper's results,
# in the same academic style as the figures above. DA-MAPPO reports dynamic
# multi-target results in Table V; its numbers below are transcribed from that
# table (ENV-1 = 30 obstacles, N=3 drones; DA-MAPPO's own success spans 90-99%
# across ENV-1..3). Our numbers come from S2_AGG / S3_AGG computed above
# (deterministic, 200 episodes). The environments are NOT the same scale
# (N=3 / 30-50 obstacles vs ours N=5-8 / 5-10 obstacles), so this is a context
# comparison, not a head-to-head -- the per-bar config labels and the caveat in
# the title make that explicit.

AMBER = '#D98E2B'   # DA-MAPPO (the base-paper method)
GREY2 = '#9AA5B1'   # literature baselines

s2_succ, s2_coll = S2_AGG['M']['success'][0], S2_AGG['M']['collision'][0]
s3_succ, s3_coll = S3_AGG['M']['success'],    S3_AGG['M']['collision']

# DA-MAPPO Table V, dynamic, ENV-1 (30 obstacles, N=3): (name, success, collision)
LIT = [('IPPO', 78, 22), ('MAPPO', 83, 17), ('RMAPPO', 85, 15),
       ('NavRL', 63, 34), ('EGO-Planner v2', 53, 45), ('DA-MAPPO', 99, 1)]

names  = [n for n, _, _ in LIT] + ['M (ours)', 'M (ours)']
cfgs   = ['N=3, 30 obs'] * len(LIT) + ['N=5, 5 obs', 'N=8, 10 obs']
succ   = [s for _, s, _ in LIT] + [s2_succ, s3_succ]
coll   = [c for _, _, c in LIT] + [s2_coll, s3_coll]
colors = [GREY2] * 5 + [AMBER] + ['#3B6FB6', '#3B6FB6']
xlabels = ['{}\n({})'.format(n, c) for n, c in zip(names, cfgs)]

def comp_panel(ax, vals, ylabel, title):
    x = np.arange(len(vals))
    bars = ax.bar(x, vals, 0.72, color=colors, edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1.2,
                '{:.0f}'.format(v) if float(v).is_integer() else '{:.1f}'.format(v),
                ha='center', va='bottom', fontsize=8.5)
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=8, rotation=25, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_ylim(0, 112)
    ax.axvline(5.5, color='#C0C0C0', lw=1, ls='--')   # separator: paper | ours

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))
comp_panel(axes[0], succ, 'Mission success rate (%)', 'Mission success rate')
comp_panel(axes[1], coll, 'Collision rate (%)',       'Collision rate')

from matplotlib.patches import Patch
handles = [Patch(fc=GREY2, label='DA-MAPPO paper, baselines (N=3, 30 obs)'),
           Patch(fc=AMBER, label='DA-MAPPO (base-paper method)'),
           Patch(fc='#3B6FB6', label='Ours -- M (PAH residual)')]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Dynamic multi-target navigation: our results vs DA-MAPPO (bin/91, Table V)\n'
             'Environments differ (drones / obstacles shown under each bar) -- context comparison, not head-to-head.',
             fontsize=10.5)
plt.tight_layout(rect=(0, 0.04, 1, 0.96))
fig.savefig(OUT + '/fig7_comparison_vs_literature.png', dpi=200, bbox_inches='tight')
fig.savefig(OUT + '/fig7_comparison_vs_literature.svg', bbox_inches='tight')
plt.close(fig)

rows = [
    '| Method | Environment (N, obstacles) | R_success (%) | R_collision (%) | Source |',
    '|---|---|---|---|---|',
    '| IPPO | N=3, 30 | 78 | 22 | DA-MAPPO Table V (ENV-1) |',
    '| MAPPO | N=3, 30 | 83 | 17 | DA-MAPPO Table V (ENV-1) |',
    '| RMAPPO | N=3, 30 | 85 | 15 | DA-MAPPO Table V (ENV-1) |',
    '| NavRL | N=3, 30 | 63 | 34 | DA-MAPPO Table V (ENV-1) |',
    '| EGO-Planner v2 | N=3, 30 | 53 | 45 | DA-MAPPO Table V (ENV-1) |',
    '| **DA-MAPPO** | N=3, 30-50 | **90-99** | 1-10 | DA-MAPPO Table V (ENV-1-3) |',
    '| **Ours -- M (Stage 2)** | N=5, 5 | **{:.1f}** | {:.1f} | this work, 5 seeds, 200 ep det. |'.format(s2_succ, s2_coll),
    '| **Ours -- M (Stage 3)** | N=8, 10 | **{:.1f}** | {:.1f} | this work, seed 42, 200 ep det. |'.format(s3_succ, s3_coll),
]
with open(OUT + '/table_comparison.md', 'w') as f:
    f.write('\n'.join(rows) + '\n')
print('Saved fig7_comparison_vs_literature (.png + .svg) and table_comparison.md to', OUT)
print('\n'.join(rows))


In [ ]:
# Cell 11 -- Comparison in the fig6 (training-curve) style, against the base paper
#
# Ayesha's ask: make the literature comparison as curves, like fig6 -- not only
# as bars (fig7). We CANNOT draw DA-MAPPO's learning curve: the paper publishes
# only final numbers (Table V), not per-episode data. Inventing their curve would
# be fabrication. Instead we plot OUR success-rate training curves (same data and
# style as fig6, Cell 9) and overlay the papers' FINAL reported success as
# horizontal reference bands, clearly labelled and at their (different) scale.
# This is orientation only, not a head-to-head -- see the caveat in the title.
#
# Reuses curve(), STYLE, GRID from Cell 9, which must be run first.

DAMAPPO_BAND  = (90, 99)   # DA-MAPPO's own success across ENV-1..3 (Table V)
BASELINE_BAND = (78, 85)   # strong MARL baselines IPPO/MAPPO/RMAPPO (Table V, ENV-1)

def add_ref_bands(ax):
    ax.axhspan(*DAMAPPO_BAND, color='#2E7D32', alpha=0.12, lw=0)
    ax.axhline(DAMAPPO_BAND[0], color='#2E7D32', lw=1.0, ls='--')
    ax.axhline(DAMAPPO_BAND[1], color='#2E7D32', lw=1.0, ls='--')
    ax.text(7950, DAMAPPO_BAND[1] - 1.5, 'DA-MAPPO reported 90-99%  (N=3, 30-50 obs)',
            ha='right', va='top', fontsize=7.5, color='#2E7D32')
    ax.axhspan(*BASELINE_BAND, color='#9AA5B1', alpha=0.18, lw=0)
    ax.text(7950, BASELINE_BAND[0] + 1.0, 'MARL baselines 78-85%  (IPPO/MAPPO/RMAPPO, N=3)',
            ha='right', va='bottom', fontsize=7.5, color='#5a6472')

with plt.rc_context({'axes.spines.top': True, 'axes.spines.right': True, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9}):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))

    ax = axes[0]
    add_ref_bands(ax)
    for m in ('B4', 'H', 'M'):
        label, colour = STYLE[m]
        mean, lo, hi = curve(2, m, 'success')
        ax.fill_between(GRID, lo, hi, color=colour, alpha=0.18, lw=0)
        ax.plot(GRID, mean, color=colour, lw=1.8, label=label)
    ax.set_ylim(0, 105); ax.set_xlim(0, 8000)
    ax.set_xlabel('Episodes'); ax.set_ylabel('Success rate (%)')
    ax.set_title('Success Rate for N=5 UAVs (Stage 2) vs reported literature')
    ax.legend(fontsize=7.5, loc='lower right', framealpha=0.9)
    ax.text(0.5, -0.17, '(a) Stage 2, 5 drones, 5 obstacles', transform=ax.transAxes,
            ha='center', va='top', fontsize=10, family='serif')

    ax = axes[1]
    add_ref_bands(ax)
    for m in ('B4', 'M', 'M-noHungarian'):
        label, colour = STYLE[m]
        mean, lo, hi = curve(3, m, 'success')
        ax.fill_between(GRID, lo, hi, color=colour, alpha=0.18, lw=0)
        ax.plot(GRID, mean, color=colour, lw=1.8, label=label)
    ax.axvline(500, color='black', lw=0.8, ls=':')
    ax.text(560, 3, 'unfreeze', fontsize=7.5, rotation=90, va='bottom')
    ax.set_ylim(0, 105); ax.set_xlim(0, 8000)
    ax.set_xlabel('Episodes'); ax.set_ylabel('Success rate (%)')
    ax.set_title('Success Rate for N=8 UAVs (Stage 3) vs reported literature')
    ax.legend(fontsize=7.5, loc='lower right', framealpha=0.9)
    ax.text(0.5, -0.17, '(b) Stage 3, 8 drones, 10 obstacles', transform=ax.transAxes,
            ha='center', va='top', fontsize=10, family='serif')

    fig.suptitle('Our success-rate training curves vs the base paper, in the fig6 style.\n'
                 'Curves = our training-time evaluation (stochastic, 20 ep/checkpoint, 5-checkpoint moving avg). '
                 'Bands = papers\' FINAL reported success at a different scale (N=3, 30-50 obstacles) -- '
                 'orientation only, not head-to-head.', fontsize=9, y=0.99)
    fig.tight_layout(rect=(0, 0.02, 1, 0.93))
    fig.savefig(OUT + '/fig8_comparison_curves.png', dpi=200, bbox_inches='tight')
    fig.savefig(OUT + '/fig8_comparison_curves.svg', bbox_inches='tight')
    plt.close(fig)
print('Saved fig8_comparison_curves (.png + .svg) to', OUT)


In [ ]:
# Cell 12 -- PAH alpha-behavior: the mission/safety weight by situation (Option A)
#
# This is PAH's actual, proven contribution: it is a tie with fixed alpha on the
# outcome, but the MECHANISM works -- it lowers alpha (safety-first) when a
# collision is imminent and raises it (mission-first) when safe. Data is the
# training-time alpha logging: eval_alpha_low_tau (alpha in danger, low time-to-
# collision) and eval_alpha_high_tau (alpha when safe). Reuses entries()/hist_path()
# from Cell 9.

def pool_alpha(stage, method, key):
    v = []
    for e in entries(stage, method):
        h = json.load(open(hist_path(e)))
        v += [x for x in h[key] if x is not None and not (isinstance(x, float) and np.isnan(x))]
    return np.array(v, dtype=float)

stages = ['Stage 2\n(5 drones)', 'Stage 3\n(8 drones)']
danger = [pool_alpha(2, 'M', 'eval_alpha_low_tau'),  pool_alpha(3, 'M', 'eval_alpha_low_tau')]
safe   = [pool_alpha(2, 'M', 'eval_alpha_high_tau'), pool_alpha(3, 'M', 'eval_alpha_high_tau')]
d_mean = [a.mean() for a in danger]; d_std = [a.std(ddof=1) for a in danger]
s_mean = [a.mean() for a in safe];   s_std = [a.std(ddof=1) for a in safe]
n_total = sum(len(a) for a in danger) + sum(len(a) for a in safe)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
x = np.arange(len(stages)); w = 0.36
b1 = ax.bar(x - w/2, d_mean, w, yerr=d_std, capsize=4, color=RED,   edgecolor='white',
            label='In danger (low time-to-collision)', error_kw=dict(ecolor='black', lw=1))
b2 = ax.bar(x + w/2, s_mean, w, yerr=s_std, capsize=4, color=GREEN, edgecolor='white',
            label='When safe (high time-to-collision)', error_kw=dict(ecolor='black', lw=1))
for bars, m, sd in ((b1, d_mean, d_std), (b2, s_mean, s_std)):
    for bar, v, e in zip(bars, m, sd):
        ax.text(bar.get_x()+bar.get_width()/2, v+e+0.02, '{:.2f}'.format(v), ha='center', va='bottom', fontsize=9)
ax.axhline(0.5, color='#444', lw=1.2, ls='--')
ax.text(-0.48, 0.52, 'Fixed-alpha baseline (B4 = 0.50)', ha='left', va='bottom', fontsize=8.5, color='#444')
ax.set_xticks(x); ax.set_xticklabels(stages)
ax.set_ylabel('Mission weight alpha   (0 = safety-first, 1 = mission-first)')
ax.set_ylim(0, 1.0)
ax.set_title('PAH learns to prioritize by situation (M):\nit lowers alpha in danger and raises it when safe')
ax.legend(loc='center', frameon=True, framealpha=0.95, fontsize=8.5)
ax.text(0.5, -0.17, 'Direction was correct in {}/{} evaluated states (0 on the wrong side of 0.5).'.format(n_total, n_total),
        transform=ax.transAxes, ha='center', va='top', fontsize=8.5, color='#333')
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT + '/fig9_pah_alpha_behavior.png', dpi=200, bbox_inches='tight')
fig.savefig(OUT + '/fig9_pah_alpha_behavior.svg', bbox_inches='tight')
plt.close(fig)
print('Saved fig9_pah_alpha_behavior. danger', [round(v,3) for v in d_mean], 'safe', [round(v,3) for v in s_mean], '| n', n_total)


In [ ]:
# Cell 13 -- Generalization to unseen conditions: M vs B4 (Option C)
#
# From stage3/generalization/generalization_results.json (deterministic eval,
# 100 episodes, held-out seeds 2001-2005; OOD = 16 obstacles, never seen in
# training). Single training run, small gap -- read as a trend, not a win.

g = json.load(open(V3 + '/stage3/generalization/generalization_results.json'))
conds   = ['In-distribution\n(seeds 2001-2005)', 'Out-of-distribution\n(16 obstacles, harder)']
m_vals  = [g['in_distribution']['M']['success'],  g['out_of_distribution']['M']['success']]
b4_vals = [g['in_distribution']['B4']['success'], g['out_of_distribution']['B4']['success']]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
x = np.arange(len(conds)); w = 0.36
bm = ax.bar(x - w/2, m_vals,  w, color='#3B6FB6', edgecolor='white', label='M (PAH residual, ours)')
bb = ax.bar(x + w/2, b4_vals, w, color=GREY,      edgecolor='white', label='B4 (fixed alpha)')
for bars, vals in ((bm, m_vals), (bb, b4_vals)):
    for j, (bar, v) in enumerate(zip(bars, vals)):
        txt = '{:.0f}%'.format(v) if j == 0 else '{:.0f}%  (-{:.0f})'.format(v, vals[0]-vals[1])
        ax.text(bar.get_x()+bar.get_width()/2, v+1.2, txt, ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_ylabel('Mission success rate (%)'); ax.set_ylim(0, 100)
ax.set_title('Generalization to unseen conditions (Stage 3):\nM holds up slightly better than fixed alpha')
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.text(0.5, -0.17, 'Deterministic eval, 100 episodes, held-out seeds. Single training run -- small gap, read as a trend not a win.',
        transform=ax.transAxes, ha='center', va='top', fontsize=8.5, color='#333')
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT + '/fig10_generalization.png', dpi=200, bbox_inches='tight')
fig.savefig(OUT + '/fig10_generalization.svg', bbox_inches='tight')
plt.close(fig)
print('Saved fig10_generalization. M', m_vals, 'B4', b4_vals)


In [ ]:
# Cell 14 -- PAH response curve: alpha as a continuous function of time-to-collision
#
# fig9 shows alpha in two buckets (danger vs safe) measured from real states.
# This shows the LEARNED FUNCTION itself: we sweep time-to-collision through the
# trained PAH head (other two inputs held at typical values) and read off alpha.
# Why a response curve and not a scatter of real eval states: the converged
# policy almost never enters danger at eval time (~99.9% of eval states have tau
# at the horizon cap -- it avoids collisions well), so a scatter has no spread in
# tau. The sweep shows what the head would output across the whole range.
#
# Needs the seed-42 M checkpoints from REGISTRY; defines PAH locally (the
# results-table Actor does not include the head).

class _PAH(nn.Module):
    def __init__(self, hidden_dim=32, alpha_min=0.1, alpha_max=0.9, delta_scale=0.2):
        super().__init__()
        self.alpha_min, self.alpha_max, self.delta_scale = alpha_min, alpha_max, delta_scale
        self.net = nn.Sequential(nn.Linear(3, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        tau_norm = x[..., 0]
        delta    = self.delta_scale * torch.tanh(self.net(x).squeeze(-1))
        return (self.alpha_min + (self.alpha_max - self.alpha_min) * tau_norm + delta).clamp(self.alpha_min, self.alpha_max)

def _load_pah(path):
    ck = torch.load(path, map_location='cpu', weights_only=True)
    p = _PAH(); p.load_state_dict(ck['pah']); p.eval(); return p

HORIZON = 3.0
@torch.no_grad()
def _response(pah, n_norm, d_norm=0.3, npts=200):
    tn = torch.linspace(0, 1, npts)
    x = torch.stack([tn, torch.full_like(tn, d_norm), torch.full_like(tn, n_norm)], dim=-1)
    return tn.numpy() * HORIZON, pah(x).numpy()

def _m_ckpt(stage):
    return [e['path'] for e in REGISTRY if e['stage'] == stage and e['method'] == 'M' and e['seed'] == 42][0]

STAGES_RC = {'Stage 2 (5 drones, 5 obstacles)': (_m_ckpt(2), 5),
             'Stage 3 (8 drones, 10 obstacles)': (_m_ckpt(3), 8)}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
for ax, (title, (ckpt, n_drones)) in zip(axes, STAGES_RC.items()):
    pah = _load_pah(ckpt)
    tline = np.linspace(0, HORIZON, 100)
    ax.plot(tline, 0.1 + 0.8 * (tline / HORIZON), color='#444', lw=1.8, ls='--', label='alpha_safe(tau): hardcoded prior', zorder=3)
    denom = max(n_drones - 1, 1)
    curves = np.array([_response(pah, min(nc, denom) / denom)[1] for nc in (0, 1, 2, 3, 4)])
    t0, a0 = _response(pah, 0.0)
    ax.fill_between(t0, curves.min(0), curves.max(0), color='#3B6FB6', alpha=0.20, lw=0, label='spread over 0-4 neighbour-conflicts', zorder=2)
    ax.plot(t0, a0, color='#3B6FB6', lw=2.6, label='PAH learned alpha (M)', zorder=4)
    ax.axhline(0.5, color='#999', lw=1.0, ls=':')
    ax.text(0.05, 0.52, 'fixed-alpha baseline 0.50', fontsize=8, color='#777')
    ax.annotate('danger\n(safety-first)', xy=(0.08, 0.14), fontsize=9, color='#C62828', ha='left', va='center')
    ax.annotate('safe\n(mission-first)', xy=(2.55, 0.86), fontsize=9, color='#2E7D32', ha='left', va='top')
    ax.set_xlabel('Time-to-collision tau  (lower = more dangerous)')
    ax.set_ylabel('Mission weight alpha'); ax.set_xlim(0, HORIZON); ax.set_ylim(0, 1.0); ax.set_title(title)
    ax.legend(loc='lower right', fontsize=7.8, framealpha=0.95)
fig.suptitle('What the Priority Arbitration Head learned: its alpha response to the situation (M).\n'
             'Sweeping time-to-collision through the trained head (other inputs at typical values). alpha rises smoothly as\n'
             'danger recedes (safety-first -> mission-first). The learned head is more safety-conservative than the prior; '
             'conflict count barely shifts it (thin band).', fontsize=9.5)
plt.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(OUT + '/fig9b_pah_response_curve.png', dpi=200, bbox_inches='tight')
fig.savefig(OUT + '/fig9b_pah_response_curve.svg', bbox_inches='tight')
plt.close(fig)
print('Saved fig9b_pah_response_curve (.png + .svg) to', OUT)
